# 🚀 YOLOv7 Custom Training on Google Colab with GPU

In [7]:
# 🔧 Clone YOLOv7 and install requirements
!git clone https://github.com/WongKinYiu/yolov7
%cd yolov7
!pip install -r requirements.txt
!pip install wandb

Cloning into 'yolov7'...
remote: Enumerating objects: 1197, done.
remote: Total 1197 (delta 0), reused 0 (delta 0), pack-reused 1197 (from 1)
Receiving objects: 100% (1197/1197), 74.23 MiB | 11.33 MiB/s, done.
Resolving deltas: 100% (520/520), done.
/content/yolov7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━

In [1]:
# ⛓️ Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 🗂️ Copy custom dataset and config from Drive
!cp -r /content/drive/My Drive/RAGChat/dataset ./dataset
!cp -r /content/drive/My Drive/RAGChat/yolov7/* ./

cp: target './dataset' is not a directory
cp: cannot stat '/content/drive/My': No such file or directory
cp: cannot stat 'Drive/RAGChat/yolov7/*': No such file or directory


In [ ]:
!pip uninstall -y tensorboard tensorflow jax jaxlib

Found existing installation: tensorboard 2.19.0
Uninstalling tensorboard-2.19.0:
  Successfully uninstalled tensorboard-2.19.0


In [ ]:
!pip install numpy==1.23.5


In [ ]:
%%writefile /content/yolov7/train.py
import os
import torch
import wandb
import yaml
from tqdm import tqdm
from yolov7.utils.loss import ComputeLoss

def train(opt, model, train_loader, val_loader, device, epochs, optimizer, log_dir):
    print("🚀 Starting YOLOv7 custom training loop with AMP and WandB")

    # Load hyperparameters
    with open(opt['hyp'], 'r') as f:
        hyp_dict = yaml.safe_load(f)

    # Assign model attributes required for YOLOv7 loss
    model.hyp = hyp_dict
    model.gr = 1.0  # objectness IoU ratio
    model.nc = model.yaml['nc']
    model.na = len(model.yaml['anchors'][0]) // 2
    model.nl = len(model.yaml['anchors'])
    model.anchors = torch.tensor(model.yaml['anchors']).float().view(model.nl, -1, 2)

    # Initialize WandB
    wandb.init(project="YOLOv7-Custom", name=opt['name'], config=opt)

    # Set up loss function and AMP
    compute_loss = ComputeLoss(model)
    scaler = torch.amp.GradScaler(enabled=(device.type != 'cpu'))

    model.train()
    best_loss = float('inf')
    os.makedirs(os.path.join(log_dir, 'weights'), exist_ok=True)

    for epoch in range(epochs):
        print(f"🔁 Epoch {epoch + 1}/{epochs}")
        total_loss = 0.0

        for batch_i, (imgs, targets, paths, _) in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch+1}", ncols=100, dynamic_ncols=True)):
            imgs = imgs.to(device).float() / 255.0
            targets = targets.to(device)

            if imgs.ndim == 3:
                imgs = imgs.unsqueeze(0)  # Ensure batch dimension

            optimizer.zero_grad()

            with torch.amp.autocast(device_type=device.type if device.type in ['cuda', 'cpu'] else 'cpu'):
                outputs = model(imgs)
                loss, loss_items = compute_loss(outputs, targets)


Overwriting /content/yolov7/train.py


In [ ]:
!cp -r /content/yolov7 /content/drive/MyDrive/yolov7_colab_backup


In [6]:
!cp /content/drive/MyDrive/yolov7_colab_backup/train.py /content/yolov7/train.py


cp: cannot stat '/content/drive/MyDrive/yolov7_colab_backup/train.py': No such file or directory


In [3]:
!head -n 25 /content/yolov7/train.py


head: cannot open '/content/yolov7/train.py' for reading: No such file or directory


In [4]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


True NVIDIA A100-SXM4-40GB


In [5]:
!python -u /content/yolov7/train.py \
  --img 640 \
  --batch 16 \
  --epochs 50 \
  --data /content/yolov7/dataset/dataset.yaml \
  --cfg /content/yolov7/cfg/training/yolov7-tiny-custom.yaml \
  --weights '' \
  --name yolov7-tiny-colab \
  --hyp /content/yolov7/data/hyp.scratch.p5.yaml



python3: can't open file '/content/yolov7/train.py': [Errno 2] No such file or directory
